# SmolVLA × LIBERO-plus Spatial Fine-tuning——最终课题教材（进阶版）

## 教材定位

请先完成 **基础版（`smolvla_libero_plus_spatial_basic.ipynb`）**，再开始进阶版。

进阶版包含：

1. 使用 LeRobot 官方实现（`SmolVLAPolicy`），对预训练 SmolVLA 做 **LoRA 追加训练**；
2. 在 LIBERO-plus **4 个 suite × 每个 3 个任务 × 1 个 episode = 每个模型 12 次 rollout**上做广域评估；
3. 通过调整 Section 8.0 的超参数，提升 LeRobot 路线性能。

#### <font color="red">希望争取贡献奖的同学，请把你的改进点发到指定 Slack。</font>

## 与基础版的关系

- **基础版**：用 PyTorch 自己实现 SmolVLA → 训练 → 评估 → 提交 `submission.json`；
- **进阶版（本 notebook）**：用 LeRobot 官方版 warm-start 训练 → 在 4 个 suite 上广域评估。

### 主要差异

- **主目标**：基础版理解 SmolVLA 内部，进阶版使用官方实现做实践；
- **模型**：基础版是教育用简化模型，进阶版是 LeRobot 官方 `SmolVLAPolicy`；
- **Action head 初始值**：基础版随机初始化，进阶版从预训练权重 warm-start；
- **训练方法**：基础版自定义 PyTorch 循环，进阶版通过 `lerobot-train` 使用 LoRA；
- **评估范围**：基础版 Spatial 2 个任务 × 1 episode，进阶版 4 个 suite × 3 个任务 × 1 episode。

### 共用的 Drive 缓存

基础版 Section 5 生成的 **LIBERO-plus assets tar.gz 缓存**（`Drive/MyDrive/smolvla_task/libero_plus_assets.tar.gz`）会被进阶版自动发现并复用。若基础版已经完成，进阶版 Section 5 只需展开 tar.gz，耗时会明显减少。

## 运行环境（Colab）

与基础版相同：使用 **GPU（T4 或更高）**并允许挂载 Drive。

## 数据与环境

- **训练数据**：`lerobot/libero_plus`（与基础版相同）；
- **评估环境**：`LiberoEnv`（LIBERO-plus 扰动仿真）；
- **评估范围**：LIBERO Spatial + Object + Goal + Long-horizon，**4 个 suite / 每个 3 个任务 / 1 个 episode = 每个模型 12 次 rollout**。


## 📖 目录与各 section 的作用

### 【准备阶段】环境与数据（与基础版共用）

1. 检查 Colab runtime——确认 GPU / Python 版本并设置 `WORKDIR`；
2. 系统包——用 apt 安装 `ffmpeg` / `libgl` 等；
3. 安装 LeRobot——clone v0.6.0 并 editable install；
4. HF 获取辅助函数——定义 429 重试和缓存优先的获取函数；
5. 准备 LIBERO-plus——复用基础版的 Drive tar.gz 缓存（没有时执行完整下载）；
6. 设置训练/评估条件——定义数据集、seed、路径等共享常量；
7. 抽取 Spatial 训练数据——等间隔选择 10 个任务 × 5 episode = 50 个。

### 【★核心】用 LeRobot 训练 → 广域评估

**Section 8（LeRobot 训练/评估）**：对预训练 SmolVLA 做 LoRA 追加训练。若要继续 Section 9，必须完整执行 Section 8。

- **8.0** 进阶超参数：调整 `STEPS` / `LORA_R` / `LEARNING_RATE` 等；
- **8.1** 下载预训练 SmolVLA（`lerobot/smolvla_libero_plus`）；
- **8.2** 用 LeRobot subprocess 进行 LoRA 训练（`lerobot-train`）；
- **8.3** 把 LoRA 合并回 base（`PeftModel.merge_and_unload`）；
- **8.4** 准备对照 baseline（用同一个 processor 保存追加训练前的权重）；
- **8.5** 在 Spatial 上比较训练前后（10 task × 1 ep × 2 model = 20 rollout）；
- **8.6** 比较 LeRobot 版本的成功率（每任务 Δ + CSV）；
- **8.7** 把合并模型压缩为 zip（提交用）；
- **8.8** 可选地输出 rollout 视频（mp4）。

**Section 9（进阶评估）**：4 suite × 3 task × 1 ep = 每模型 12 次 rollout（共 24 次）。


## 1. 检查 Colab runtime

请先把 Colab runtime 切换到 GPU，再执行此 cell。


In [ ]:
# ==============================================================
# 环境变量与运行前检查 + Google Drive 挂载
# ==============================================================
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- Google Drive 挂载 (持久化产物) --------------------
# Colab 的 /content/ 运行时结束后该目录会被清除，因此、submission.json 和
# 训练好的模型可以 Drive 保存到 Drive，因此先挂载.
# 会出现认证弹窗，请在浏览器中允许 (首下一次).
from google.colab import drive

drive.mount("/content/drive")

# Drive 复制产物到 Drive 的位置. `MyDrive/smolvla_task/` 配下.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/smolvla_task")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"DRIVE_BACKUP_DIR: {DRIVE_BACKUP_DIR}")

# --- 工作目录 ------------------------------------------
# 为优先运行速度，使用 /content/ (ephemeral) 到置. Drive 直接在 Drive 上操作会使
# 45 万个小文条的展开变慢. 仅最终产物 Drive 复制到其中.
WORKDIR = Path("/content/workdir")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"WORKDIR         : {WORKDIR}")

# --- Hugging Face Hub 的抑制噪声 ------------------------------
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
# HF / LeRobot 的缓存固定到 WORKDIR 目录下
os.environ["HF_HOME"] = str(WORKDIR / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKDIR / "lerobot_cache")
# 长时间训练中容易出现的碎片化 OOM 以避免
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LeRobot v0.6.0 是 Python 3.12 以上是必需的
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上是必需的。")

# GPU 必須
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU请选择 GPU runtime "
        "(Colab 菜单: runtime → 更改 runtime 类型 → GPU)"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 准备系统包

安装 LeRobot、视频解码和 MuJoCo 所需的系统包。


In [ ]:
# ==============================================================
# subprocess 执行辅助函几 + apt 并安装系列列统包
# ==============================================================


# 外部命令， stdout/stderr 统一捕获后执行的辅助函几:
#   - 成功时不输出日志， notebook 避免污染 notebook
#   - 失败时只把末尾 6000 个字符放入异常以便查看
#   - stderr 是 stdout 合并到 stdout 以保留顺序
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


# 用于长时间命令: 1 每收到一行 print 执行 streaming 版个个。
# 看不到进度 = 避免被误认为无限循环的辅助函几。
# - prefix 加上前缀以标明命令输出
# - 最近的日志 200 行保存在环形缓冲区中，失败时用于转储
def run_streaming(
    command: list[str],
    *,
    prefix: str = "",
    check: bool = True,
) -> int:
    from collections import deque

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    recent: deque[str] = deque(maxlen=200)
    assert process.stdout is not None
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        print(f"{prefix}{line}", flush=True)
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# 用于长时间命令 (pip install 等输出达到几百行的情况):
# 直近 tail 仅最近几行 ipywidgets 的 HTML box ， "上書更新" 显示执行.
# 显示完整时间线会使 notebook 膨胀，因此、只显示末尾.
def run_streaming_rolling(
    command: list[str],
    *,
    prefix: str = "",
    tail: int = 10,
    check: bool = True,
) -> int:
    from collections import deque
    import ipywidgets as widgets
    from IPython.display import display

    recent: deque[str] = deque(maxlen=tail)
    _widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_widget)

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    _count = 0
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        _count += 1
        # 逐行更新会使 UI 变慢，因此 5 每若干行更新 widget
        if _count % 5 == 0:
            _widget.value = (
                "<pre style='margin:0;font-size:11px;color:#666;"
                "line-height:1.3'>"
                + "\n".join(f"  {prefix}{l}" for l in recent)
                + "</pre>"
            )
    # 反映到最后一行
    _widget.value = (
        "<pre style='margin:0;font-size:11px;color:#666;line-height:1.3'>"
        + "\n".join(f"  {prefix}{l}" for l in recent)
        + "\n  ✅ done ({_count} lines total)".replace("{_count}", str(_count))
        + "</pre>"
    )
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# apt 缓存更新
run_quiet(["sudo", "apt-get", "update", "-qq"])
# 依赖包说明:
#   ffmpeg                : LeRobot 的视频解码
#   git / unzip           : 个 notebook 中 clone / 用于展开
#   libgl1 / libglib2.0-0 : OpenGL 系列列（MuJoCo/robosuite 的渲染）
#   libsm6 / libxext6     : X11 系列列列 runtime（渲染库依赖）
#   libexpat1             : XML 解析器（MuJoCo 的 XML 模型加载）
#   libfontconfig1-dev    : matplotlib 用字体设置
#   libmagickwand-dev     : Wand（ImageMagick binding） LIBERO 经由它需要
run_quiet(
    [
        "sudo", "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. 安装 LeRobot

使用 LeRobot `v0.6.0`，本 cell 同时应用 Colab LoRA 训练所需的兼容性调整。


In [ ]:
# ==============================================================
# LeRobot v0.6.0 的 editable install 与兼容性补丁
# ==============================================================
# 手順:
#   1. 现有 install 的除去（torchao 也同时。SmolVLA 因为不支持）
#   2. 固定 tag 的 LeRobot  shallow clone（进度 stream）
#   3. bf16 不支持 GPU 若 SmolVLM  fp16 改写为
#   4. lerobot_train.py 的冗长日志和 tqdm 用轻量补丁抑制
#   5. editable install（training + smolvla + peft 的 extras）— 输出 stream
#   6. torchao 重新次卸载（防止依赖重新安装的保险措施）
#   7. sys.modules 与 sys.path 掃除并 clone 完成 src 设为优先
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKDIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

# --- 0. 安全部部保护: torch 的版个个个一致性检查 ------------
# 上一个 session 中 torch  import 已导入，之后 pip install 
# torch 更新后、C 扩展(.so)是旧的的版个个，・Python 侧为新版个个的
# 不一致状态会导致 "cannot import name '_EvalFrameOverride'" 等问题。
# 事先比较磁盘上的版个个个与内存中的 in-memory 的 torch 版个个个、
# 若不同则提示重启 kernel。
# torch.__version__ / importlib.metadata.version("torch") 在不同环境中可能
# '2.11.0+cu128' 的到 CUDA 标签 (local version identifier) 可能带有
# 也可能没有. 从两侧都去除 '+' 之后的内容 PEP 440 的 public version 后重新比较.
def _strip_local_version(v: str) -> str:
    return v.split("+", 1)[0]

_torch_version_in_memory = _strip_local_version(torch.__version__)
try:
    _torch_dist_version = _strip_local_version(
        importlib.metadata.version("torch")
    )
except importlib.metadata.PackageNotFoundError:
    _torch_dist_version = None

if _torch_dist_version and _torch_dist_version != _torch_version_in_memory:
    raise RuntimeError(
        f"torch mismatch: in-memory={_torch_version_in_memory} "
        f"on-disk={_torch_dist_version}. "
        "kernel重新起動并的cell最初从执行直并请。"
    )

# --- 1. 现有 install 移除（为了安全部部重复执行） ----------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

# --- 2. LeRobot  shallow clone（进度 stream 显示） -----------
# 网络慢时需要几分钟，但、输出很多，所以只显示最近 10 行
print("Cloning LeRobot...", flush=True)
run_streaming_rolling(
    [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ],
    prefix="[git] ",
    tail=10,
)

# --- 3. dtype 兼容性补丁 -------------------------------------
# T4/V100 系列列等 bf16 不支持 GPU ，是 bf16 ， load 会导致损坏，因此 fp16 替换为
smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

# --- 4. lerobot_train.py 的軽微补丁 ---------------------------
# 训练 subprocess 的压缩日志:
#   - 配置 pformat 的 INFO  DEBUG 到降低
#   - inside_slurm() 不论判定结果 tqdm 强制 disable
train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

# --- 5. editable install（extras 包含） ------------------------
# --upgrade 不添加: torch 如果包含它的依赖已经满足，
# 就不改动任何内容（现有的 C 以避免与 C 扩展不一致）。
# 解析依赖和构建需要 5〜20 分钟. pip 的输出是几百行会变成的， rolling 显示 (直近 10 行仅).
print("Installing LeRobot (this may take a while)...", flush=True)
run_streaming_rolling(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ],
    prefix="[pip] ",
    tail=10,
)

# --- 5.5. 安装后 torch 重新次确认运行正常 --------
_torch_dist_version_after = _strip_local_version(
    importlib.metadata.version("torch")
)
if _torch_dist_version_after != _torch_version_in_memory:
    raise RuntimeError(
        f"pip install  torch  {_torch_version_in_memory} → "
        f"{_torch_dist_version_after} 已从旧的版个个变更为新版个个。"
        "kernel重新起動并的cell最初从执行直并请。"
    )

# --- 6. torchao 重新卸载 ------------------------------
# 上的 install 可能在安装中作为依赖恢复，因此重新次删除
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

# --- 7. sys.modules 与 sys.path 的掃除 --------------------------
# 上次执行 cell 时加载的旧的的 lerobot/torchao 模块移出缓存
for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

# clone 完成 src 以优先解析 sys.path 替换
sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

# torchao 检查是否完全部部删除。如果还存在， SmolVLA 会在启动时失败
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchao的删除到失败完成。")

# 实际 import 并、read 路径位于当前 clone 完成 src 目录下
# 首次 import 是 torch/CUDA 初始化可能需要 30〜60 秒
print("Importing lerobot (torch init may take up to a minute)...", flush=True)
import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobot的加载路径不正确。")

print("LeRobot ready.")

## 4. 准备公开文件获取流程

优先使用缓存；匿名访问受限时自动重试。


In [ ]:
# ==============================================================
# Hugging Face 获取重试 + 缓存优先 wrapper
# ==============================================================
# 匿名称称访问容易触发 429 (Too Many Requests) ，因此实现指几退避
# 、若已有缓存则跳过实际 DL 以避免。
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


# 429 最多 6 次重试。其他错误立即重新抛出。
def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            # 429 以外不重试（404 不反复请求）
            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            # 如果服务器返次 Retry-After 则遵守它、否则使用指几退避
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Face获取失败。"
    ) from last_error


# 先 local_files_only 检查缓存、若没有则实际 DL 执行。
# 大型模型/避免重复获取几据集的优化。
def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        # 命中缓存时不访问网络并立即返次
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        # 缓存未命中 → 429 带重试实际 DL
        # max_workers=1 以温和方式访问服务器（429 次避）
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 5. 准备 LIBERO-plus 评估环境

安装 MuJoCo / robosuite / LIBERO-plus fork，并展开评估所需 assets。这个步骤同时服务 PyTorch 版评估（section 10）与 LeRobot 版评估（section 8），请尽早完成。

由于需要下载并展开约 6GB 的 `assets.zip`，这个 cell 会花较长时间。


In [ ]:
# ==============================================================
# LIBERO-plus 设置评估环境
# ==============================================================
# 手順:
#   1. MuJoCo/robosuite 系列列的依赖 pin 带有 install
#   2. LIBERO-plus fork 固定 SHA ， clone/checkout
#   3. assets.zip  HF 从其获取并展开
#   4. ~/.libero/config.yaml 書asset位置教
#   5. lerobot_eval.py 到进度显示用的补丁当
#   6. sys.path / sys.modules 掃除并 fork 側优先加载
from huggingface_hub import hf_hub_download
import time

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORKDIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
# assets (45 万小文条 6GB) 的保存先. /content/ 配下因此展开使用高速.
LIBERO_PLUS_ASSETS_DIR = WORKDIR / "libero_plus_assets" / "assets"

# Colab 的无头 GPU 环境，是 EGL 后端， MuJoCo 描画
os.environ["MUJOCO_GL"] = "egl"

# --- 1. 现有 install 移除（重新执行安全部部性） --------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

# --- 2. LIBERO-plus 依赖的 pin 付 install --------------------
# -揃与 robosuite / mujoco 的 API 差，崩溃执行为了厳格到固定
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",  # LIBERO-plus 预期执行固定版个个
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
        "future",  # bddl 1.0.1 依赖宣言并隠依赖
    ]
)

# robosuite 预期的版个个加载被与担保
if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

# --- 3. LIBERO-plus fork  clone -----------------------------
if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

# 固定 SHA 方向 checkout。现有 clone 到対并也重新执行安全部部会变成、
# checkout 失败時仅 fetch ，恢复执行
checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    # SHA -课程到没有（shallow clone 等）若 fetch 并从 checkout
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

# fork  editable install。--no-deps ， pin 済的依赖覆盖更新被
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

# --- 4. assets.zip  HF 从 DL・展开 -------------------------
# 大 asset 是 fork 側到含的， Sylvest/LIBERO-plus 从获取
if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORKDIR / "libero_plus_assets"
    assets_root.mkdir(parents=True, exist_ok=True)

    # ---- Drive 缓存策略 ----
    # 优先级: (1) tar.gz 缓存 (最快)  > (2) zip 缓存  > (3) HF DL
    # tar.gz  Drive 保存后、下次 session 从恢复到展开只需几分钟.
    _drive_tar = DRIVE_BACKUP_DIR / "libero_plus_assets.tar.gz"
    _drive_zip = DRIVE_BACKUP_DIR / "libero_plus_assets.zip"

    if _drive_tar.is_file():
        # ---- 最快路径: tar.gz  Drive 从 Drive 取出并展开 ----
        print(
            f"Drive 的 tar.gz 正在从缓存恢复 (unzip 无需 unzip): {_drive_tar}",
            flush=True,
        )
        _t0 = time.time()
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar

        # tar.gz 是 zip 不同，它没有中央目录，因此、
        # 通常需要流式扫描才能知道文条总几.
        # 因此使用上次的 tar 创建时 .count 把几量写入 sidecar、
        # 如果存在就用它 tqdm 的 total 立即填充 total.
        _drive_tar_count = _drive_tar.with_suffix(_drive_tar.suffix + ".count")
        _tar_total = None
        if _drive_tar_count.is_file():
            try:
                _tar_total = int(_drive_tar_count.read_text().strip())
                print(f"sidecar从 sidecar 获取几量: {_tar_total} files", flush=True)
            except Exception:
                _tar_total = None
        if _tar_total is None:
            # 无 sidecar (旧的的缓存): getmembers() ， 1 扫描一次.
            # data 不展开几据，所以比完整展开快，但、仍需等待几十秒.
            print("正在统计文条几 (首下一次、需要一点时间)...", flush=True)
            with tarfile.open(_drive_tar, "r:gz") as _tf:
                _tar_total = sum(1 for _ in _tf)

        with tarfile.open(_drive_tar, "r:gz") as _tf:
            _pbar_tx = _tqdm_tar(total=_tar_total, desc="tar 展开", unit="files")
            while True:
                _m = _tf.next()
                if _m is None:
                    break
                _tf.extract(_m, assets_root)
                _pbar_tx.update(1)
            _pbar_tx.close()
        print(f"tar 展开完成 (in {time.time()-_t0:.0f}s)", flush=True)

        # 旧的的 zip 缓存固定到删除 (Drive 容量节省)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧的 zip 缓存固定到删除: {_drive_zip}", flush=True)
            except Exception:
                pass
        # → LIBERO_PLUS_ASSETS_DIR 应当存在. 以下 DL/unzip 跳过.
        _skip_dl_unzip = True
    else:
        _skip_dl_unzip = False

if not LIBERO_PLUS_ASSETS_DIR.is_dir() and not _skip_dl_unzip:
    # ---- 中速路径: zip 缓存从恢复 or HF DL ----
    local_zip = assets_root / "assets.zip"

    if _drive_zip.is_file() and not local_zip.is_file():
        print(f"Drive 缓存从 assets.zip 恢复: {_drive_zip}", flush=True)
        shutil.copy2(_drive_zip, local_zip)

    if local_zip.is_file():
        archive_path = local_zip
        print(f"现有的 assets.zip 使用: {archive_path}", flush=True)
    else:
        print("HF 从 assets.zip 正在下载 (6GB, 需要时间)...", flush=True)
        # ---- HF DL 期间 progress bar 启用 ----
        # env-var 的 pop 是 huggingface_hub  import 时 disabled 因为被缓存所以不起作用.
        # 官方文档的 API enable_progress_bars() / disable_progress_bars() 显式切换.
        from huggingface_hub.utils import (
            enable_progress_bars,
            disable_progress_bars,
        )
        enable_progress_bars()
        try:
            _t0 = time.time()
            archive_path = Path(
                run_hf_with_retry(
                    lambda: hf_hub_download(
                        repo_id="Sylvest/LIBERO-plus",
                        repo_type="dataset",
                        filename="assets.zip",
                        local_dir=assets_root,
                        token=False,
                    )
                )
            )
            _dl_sec = time.time() - _t0
        finally:
            disable_progress_bars()

        _size_mb = archive_path.stat().st_size / 1e6
        print(
            f"HF DL 完成: {_size_mb:.0f} MB in {_dl_sec:.0f}s "
            f"({_size_mb / max(_dl_sec, 1):.1f} MB/s)",
            flush=True,
        )
        # 注意意: 在此阶段 Drive 到 zip 書还是.
        # 展开后到 tar.gz 创建 Drive 到置 (下面的处理) 更
        # 也能更快、更省空间地从下次 session 恢复.
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 展开使用 Python 的 zipfile + tqdm progress bar 执行.
    # ・整体进度用 tqdm 进度条显示 (ETA / files/sec 包含).
    # ・直近 10 文条名称称 ipywidgets 的 HTML box ， "上書更新" 执行
    #   (subprocess 的 unzip -o 会导致 45 万行输出并让 notebook 卡死，因此).
    import zipfile
    import ipywidgets as widgets
    from IPython.display import display
    from tqdm.auto import tqdm
    from collections import deque

    print(f"Extracting {archive_path.name} to {extract_dir}...", flush=True)

    # 最近展开的文条 10 个文条的 widget
    _recent_paths: deque[str] = deque(maxlen=10)
    _recent_widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_recent_widget)

    _t0 = time.time()
    with zipfile.ZipFile(archive_path) as _zf:
        _members = _zf.infolist()
        _pbar = tqdm(_members, desc="unzip", unit="files")
        for _i, _m in enumerate(_pbar):
            _recent_paths.append(_m.filename)
            _zf.extract(_m, extract_dir)
            # 500 每个文条 widget 覆盖更新 (每次更新会很慢)
            if _i % 500 == 0:
                _recent_widget.value = (
                    "<pre style='margin:0;font-size:11px;color:#666;"
                    "line-height:1.3'>"
                    + "\n".join(f"  {p}" for p in _recent_paths)
                    + "</pre>"
                )
        # 最终状态
        _recent_widget.value = (
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>"
            + f"  ✅ done: {len(_members)} files extracted"
            + "</pre>"
        )
    print(f"unzip 完成 (in {time.time()-_t0:.0f}s)", flush=True)

    # zip 中的目录层级可能变化，因此 rglob ， assets/ 搜索。
    # 优先选择嵌套较浅的路径（真实的 assets 预计更接近根目录）
    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    # 展开后的真实 assets  fork 移动到 fork 内的预期位置
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    # assets 移动完成，因此 extract 只删除临时目录。
    shutil.rmtree(extract_dir, ignore_errors=True)

    # ---- 为下次 session tar.gz  Drive 缓存到 ----
    # zip + unzip 的完整流程很慢，因此、展开后的 assets  tar.gz 化并 Drive 保存到.
    # 下次 session 中 tar 只需展开、几分钟即可恢复.
    try:
        _t0 = time.time()
        print(
            f"Drive 到 tar.gz 正在创建缓存...",
            flush=True,
        )
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar_c

        # 创建时可以预先统计文条几，因此 tqdm 的 total 可以传入 total
        _entries = list(LIBERO_PLUS_ASSETS_DIR.rglob("*"))
        _total_entries = len(_entries) + 1  # +1: root dir 个身
        _pbar_tc = _tqdm_tar_c(total=_total_entries, desc="tar 创建", unit="files")

        def _tar_filter(tarinfo):
            _pbar_tc.update(1)
            return tarinfo

        with tarfile.open(_drive_tar, "w:gz", compresslevel=1) as _tf:
            _tf.add(LIBERO_PLUS_ASSETS_DIR, arcname="assets", filter=_tar_filter)
        _pbar_tc.close()
        # 下次恢复时 tqdm 的 total 以便立即显示、保存文条几 sidecar.
        _drive_tar.with_suffix(_drive_tar.suffix + ".count").write_text(
            str(_total_entries) + "\n"
        )
        print(f"tar.gz 缓存完成 (in {time.time()-_t0:.0f}s)", flush=True)
        # 旧的的 zip 缓存固定到删除 (Drive 容量节省)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧的 zip 缓存固定到删除: {_drive_zip}", flush=True)
            except Exception:
                pass
    except Exception as _e:
        print(f"tar.gz 创建缓存失败 (忽略并继续): {_e}", flush=True)

# --- 5. ~/.libero/config.yaml 写入 --------------------------
# LIBERO 启动时读取此文条并解析 assets/bddl/datasets/init_files 的
# 位置。fork 指向 fork 侧路径
# ---- LIBERO 所期望的 assets 路径的 symlink 创建 ----
# LIBERO 的 robosuite Arena XML 加载使用 fork 内的相对路径
# ({LIBERO_PLUS_DIR}/libero/libero/assets/...) 硬编码引用.
# 另一方面，个 notebook 把 notebook 是 45 万个小文条实体放在 LIBERO_PLUS_ASSETS_DIR 中.
# 两者通过 symlink 连接后、LIBERO 是 "预期位置" 即可找到实体.
_libero_expected_assets = LIBERO_PLUS_DIR / "libero" / "libero" / "assets"
if _libero_expected_assets.is_symlink() or _libero_expected_assets.exists():
    if _libero_expected_assets.is_symlink():
        _libero_expected_assets.unlink()
    else:
        shutil.rmtree(_libero_expected_assets, ignore_errors=True)
_libero_expected_assets.parent.mkdir(parents=True, exist_ok=True)
_libero_expected_assets.symlink_to(LIBERO_PLUS_ASSETS_DIR)
print(f"symlink: {_libero_expected_assets} -> {LIBERO_PLUS_ASSETS_DIR}")

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

# --- 6. lerobot_eval.py 应用进度显示补丁 -----------------
# 默认评估脚个会输出冗长的 log 与 tqdm ，所以抑制它。
# 改为 "EVAL_PROGRESS task=x/y episode=a/b" 格式 stdout 写到 stdout、
# 由个 notebook 解析并显示。
eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

# 冗长的配置 pformat  DEBUG 到降低
source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
# 在非录制时 rollout 视频最大 10 个 rollout 视频的部分固定为 0 到固定
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
# tqdm 强制禁用
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

# 当前 task/episode 把保存进度的全部部局字典注意意入脚个
progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

# task 在循环开头 _EVAL_PROGRESS 插入更新代码
task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

# episode 在循环每轮输出进度 1 行 stdout 写出到 stdout
episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

# --- 7. sys.path 清理并验证 fork 的加载 ---------------------
# pip install 完成 LIBERO-plus fork 确保绝对优先
libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

# 旧的的 LIBERO/robosuite 清除模块缓存
for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

# libero 包的加载路径位于 fork 目录下
search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

# benchmark 模块也做同样检查
benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 6. 设置训练与评估条件

这里仅定义**两条路线共有的内容**：

- 模型 / 数据集 repo 与 revision；
- LIBERO-Spatial 的 10 个任务列表；
- 训练使用的 episode 数和 seed；
- 输出路径与混合精度。

每条路线自己的超参数在下面定义：
- PyTorch 版 → `section 9.3` 的 `TRAIN_STEPS_PY` 等；
- LeRobot 版 → Advanced 的 `STEPS` 等。


In [ ]:
# ==============================================================
# 训练・集中定义评估使用的常量和路径
# ==============================================================

# --- 模型 / 固定几据集 revision -----------------------
# revision  commit hash 固定以保证可复现
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

# SmolVLA 的 Vision-Language backbone（500M 参几版个个个）
VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

# --- LIBERO-Spatial 的 10 任务（训练・评估共用） -------------
# 几据集側的 task 名称称称必须严格一致
SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

# --- 训练・选择评估几据 --------------------------------------
TRAIN_EPISODES_PER_TASK = 5   # 10 tasks × 5 = 50 episodes 选择
SEED = 42                     # 随机 seed
EVAL_SEED = 2026              # 评估 seed

# --- 输出路径 --------------------------------------------------
OUTPUT_DIR = WORKDIR / "outputs" / "smolvla_libero_plus_spatial_lora"
MERGED_MODEL_DIR = WORKDIR / "smolvla_libero_plus_spatial_lora_merged"
BASELINE_MODEL_DIR = WORKDIR / "smolvla_libero_plus_baseline"

BASE_EVAL_DIR = WORKDIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORKDIR / "eval" / "spatial_lora"
COMPARISON_CSV_PATH = WORKDIR / "libero_spatial_comparison.csv"
MERGED_ZIP_PATH = WORKDIR / "smolvla_libero_plus_spatial_lora_merged.zip"

# --- 混合精度模式 --------------------------------------------
# bf16 支持 GPU（A100/H100/L40/RTX40 系列列等）若 bf16、否则 fp16
MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

## 7. 选择 Spatial 训练数据

从 10 个任务中为每个任务等间隔选择 5 个 episode。


In [ ]:
# ==============================================================
# LIBERO-Spatial 10 从每个任务选择 5 episode 等间隔选择
# ==============================================================
# 可复现地构建训练子集。
# 几据集側的 task 名称称称可能有变体，因此用规范化 key 匹配、
# 每个 task 的 episode index 从等间隔到 5 个样个（合计 50 episode）。
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


# 转小写・移除符号・规范化空白以生成匹配 key。
def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# 元几据的 tasks 列可能是 str/list 两者都可能，因此转成单一 str 到降低。
def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


# episode 从集合中等间隔选择 count 个（始末两端都包含）。
def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    if len(episode_indices) < count:
        raise ValueError(
            f"Task 中只有 episode  {len(episode_indices)} 条，因此、"
            f"要求几量 {count} 无法满足。"
        )
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


# 获取几据集元几据（429 带重试）
dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

# 「task 名称称 → 该 task 所属的 episode index 列表」构建
task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

# 归一化-从実際的任务名称称（包含名称称称变体）用于反查的字典
available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

# Spatial 的 10 从每个任务 5 episode 选择
selected_by_task: dict[str, list[int]] = {}

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )

    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )

    selected_by_task[actual_task] = choose_evenly_spaced(
        task_to_episodes[actual_task],
        TRAIN_EPISODES_PER_TASK,
    )

# 扁平的 episode index 汇总为排序后的列表
EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

# 预期: 10 tasks × 5 = 50 个
if len(EPISODE_INDICES) != 50:
    raise RuntimeError("Episode selection failed.")

print("Training data: 10 tasks × 5 episodes = 50 episodes")

## 8. 使用 LeRobot 进行 LoRA 追加训练

> **📌 如果要进入 Section 9，本 section 必须完成。**
> Section 9 的进阶评估需要 Section 8.3 生成的合并模型和 8.4 生成的对照 baseline。不能只执行 Section 9，请先完成整个 Section 8。

> **📎 本 section 负责生成 Section 9 的输入。**
> 这里生成的训练后模型（`MERGED_MODEL_DIR`）和对照 baseline（`BASELINE_MODEL_DIR`）会作为 Section 9 的输入。只要经由 `SmolVLAPolicy.save_pretrained` 保存，传入 Section 9 的格式就会自动正确；通常只需调整 Section 6 的超参数或 Section 8.2 的 `lerobot-train` 参数，评估代码无需修改。详见 Section 9 开头的“输入格式契约”。

> **📌 LoRA 的优势在这里体现。** 基础版从随机初始值训练 Action head，因此 LoRA 的收益有限；进阶版先 warm-start `lerobot/smolvla_libero_plus` 的**预训练 Action head**，再学习少量差分，这才是 LoRA“保留预训练，只学习小幅更新”的典型用法（`--peft.method_type=LORA --peft.r=16`）。

基础版用 PyTorch 从零写出了教育版 SmolVLA；LeRobot 已经提供 `SmolVLAPolicy`、预训练权重以及 `lerobot-train` / `lerobot-eval` CLI。使用它们可以 warm-start 预训练 Action head，让训练从较好的起点开始；训练在 subprocess 中运行，notebook 只负责等待；评估也能通过 CLI 在 Spatial 10 个任务上运行。

Section 8.1–8.6 完成追加训练和 Spatial 对比评估，8.7–8.8 生成 zip 与视频，随后进入 Section 9 的 4-suite 广域评估。


### 8.0 LeRobot 版超参数

在这里定义 LeRobot subprocess（`lerobot-train` / `lerobot-eval`）使用的超参数。

### 🛠️ Section 8 中可以尝试的改动

只要保持 `SmolVLAPolicy.save_pretrained` 的输出格式，下面的改动都能直接由 Section 9 评估：

- **LoRA rank / alpha**：修改 `LORA_R`、`LORA_ALPHA`，权衡表达能力和可训练参数量；
- **训练 step 数**：修改 `STEPS`，影响收敛程度；
- **学习率 schedule**：调整 `LEARNING_RATE`、`FINAL_LEARNING_RATE`、`WARMUP_STEPS`，影响稳定性；
- **batch size**：调整 `BATCH_SIZE`，注意显存；
- **seed**：修改 Section 6 的 `SEED`，改变训练初始条件；
- **LoRA 目标模块**：在 Section 8.2 的 `lerobot-train` 参数中添加 `--peft.target_modules`；
- **训练 episode 抽取**：改变 Section 7 的 `EPISODE_INDICES`。

### ⚠️ 容易破坏契约的改动

以下改动可能让 Section 9 无法运行：

- 把 `BASE_MODEL_REPO` 改成非 SmolVLA（例如 pi0）：8.3 的 merge 写死了 SmolVLAPolicy；
- 添加未被 `save_pretrained` 保存的自定义 `nn.Module`；
- 手工把 `MERGED_MODEL_DIR` 改成另一种格式，造成文件缺失；
- 把 `train_expert_only=False`，会训练 VLM，增加时间和显存并降低稳定性；
- 改 `chunk_size` 却不让评估与 LIBERO env 的 fps 保持一致；
- 改图像尺寸却不同步 Section 9 的 `--env.observation_height/width`。

因此建议先从 8.0 的超参数改动开始，并理解其影响。


In [ ]:
# ==============================================================
# 8.0 LeRobot 版个个超参几
# ==============================================================
# LeRobot subprocess (lerobot-train / lerobot-eval) 使用的超参几.

# ---- LoRA 追加训练用 (section 8.2 的 lerobot-train 使) ----
STEPS               = 3000        # 训练总 step 几
LOG_FREQ            = 100         # loss print 频率
BATCH_SIZE          = 1           # mini-batch 大小 (VRAM 按显存调整)
LEARNING_RATE       = 3e-4        # peak learning rate
FINAL_LEARNING_RATE = 3e-5        # cosine decay 后的 lr
WARMUP_STEPS        = 100         # linear warmup step 几
LORA_R              = 16          # LoRA 低秩维度 (4/8/16/32 是常见设置)
LORA_ALPHA          = 16          # LoRA 缩放 (通常 = r or 2r)

# ---- LeRobot 评估用 (section 8.5 的 lerobot-eval 使) ----
EVAL_TASK_IDS          = list(range(10))   # LIBERO-Spatial 的 task_id (全部部 10)
# Section 8.5 是 10 task × 1 ep × 2 model = 20 rollout
EVAL_EPISODES_PER_TASK = 1

# 打印当前值 (确认使用哪些超参几训练/评估)
print("[section 8.0] 使用的超参几:")
print(f"  learning: STEPS={STEPS}, LR={LEARNING_RATE}, WARMUP={WARMUP_STEPS},")
print(f"            BATCH={BATCH_SIZE}, SEED={SEED}")
print(f"  LoRA    : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  eval    : TASK_IDS={EVAL_TASK_IDS},")
print(f"            EPISODES_PER_TASK={EVAL_EPISODES_PER_TASK}, SEED={EVAL_SEED}")

### 8.1 下载预训练 SmolVLA


In [ ]:
# ==============================================================
# 下载追加训练的初始权重
# ==============================================================
# 只获取需要的最小文条。README 和 eval/**（视频文条）不下载。
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",                        # 模型结构
        "model.safetensors",                  # 个体 weight
        "train_config.json",                  # 训练配置（以便复用）
        "policy_preprocessor.json",           # 输入预处理配置
        "policy_preprocessor*.safetensors",   # 输入侧 stats
        "policy_postprocessor.json",          # 输出后处理配置
        "policy_postprocessor*.safetensors",  # 输出侧 stats
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",  # 公开版个个 repo 中包含的评估视频（很大）不需要
    ],
)

# 个体 weight 确认已可靠获取
if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

### 8.2 用 LeRobot subprocess 进行 LoRA 训练

每 100 step 显示一次平均 loss 和 learning rate。


In [ ]:
# ==============================================================
# lerobot-train  subprocess 启动并实时显示进度
# ==============================================================
# lerobot-train 的 CLI 直接调用 LoRA 执行训练。
# stdout 解析 step/loss/lr  LOG_FREQ 每个 1 只显示一行。
import re
from collections import deque

# EPISODE_INDICES  CLI 作为参几传递 JSON 格式化为几组字符串
episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

# --- lerobot-train 的 CLI 参几 ---------------------------------
command = [
    "lerobot-train",
    # 初始权重与 VLM backbone 的路径
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    # Hub 到 push 不执行
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    # 入输出 features 尊重 checkpoint 中的值
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    # LoRA 目标只包括 expert 侧。VLM 侧冻结
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    # 学习率和 cosine scheduler
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    # 几据集和选出的 episode
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    # 输出目录 / job 名称称称
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_spatial_lora",
    # 训练步几 / batch / DataLoader
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",            # subprocess 中的追加 worker 禁用
    "--persistent_workers=false",
    # 训练期间的 env 评估禁用（由个 notebook 另外评估）
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    # checkpoint 只保存最终 step 到个地
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    # LoRA 配置
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

# --- 子进程环境变量 ---------------------------------------
training_env = os.environ.copy()
# clone 完成 src  PYTHONPATH 放在前面以使用已打补丁的 LeRobot 使其使用
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
# accelerate 向它传入混合精度模式（bf16/fp16）
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
# stdout 按行缓冲以便实时显示进度
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

# 从干净的输出目录开始
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

# --- 启动 subprocess 并 stdout 逐行读取 --------------------
process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 最近的日志 80 保留行、失败时转储（用于诊断）
recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

# "step:... loss:... lr:..." 提取行并格式化显示
for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(r"loss:([0-9.eE+-]+)", line)
        lr_match = re.search(r"lr:([0-9.eE+-]+)", line)

        loss = loss_match.group(1) if loss_match else "n/a"
        lr = lr_match.group(1) if lr_match else "n/a"

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

# 失败时先输出末尾日志，重新 raise（用于排查原因）
if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

### 8.3 将 LoRA 合并回 base

把 LoRA 差分合并进原始权重，并保存为普通 LeRobot 模型。


In [ ]:
# ==============================================================
# LoRA adapter 合并到 base weight 并转换为单体模型
# ==============================================================
# 训练后的 adapter  PeftModel 加载 → merge_and_unload ，
# LoRA 差分加到 base weight 到足包含、普通的 LeRobot 模型格式保存。
# 这样评估时 peft 无需经过 adapter 即可走原生推理路径。
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

# lerobot-train 保存的最终 checkpoint（adapter 位于其中）
checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

# GPU 释放内存后 CPU 上进行 merge（VRAM 节省）
gc.collect()
torch.cuda.empty_cache()

# --- - + LoRA 加载设置 ---------------------------------
merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"                     # merge 使用 CPU ，执行
merge_config.pretrained_path = BASE_MODEL_LOCAL # - weight 明确位置
merge_config.use_peft = False                   # merge 后 peft 不使用该路径

# 加载时的 noisy  info 静默日志
quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 1) - SmolVLA  CPU 加载到
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    # 2) 训练好的 adapter 覆盖
    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    # 3) LoRA  base 加次并转换为单体模型
    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

# --- 保存合并模型 -------------------------------------
shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config 是「单体模型・重新 push 无・VLM 不重新加载」到固定
merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

# 训练期间更新的预处理/后处理 stats 也必须复制到同一目录，否则
# 评估时输入输出缩放会失效
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

# --- 在 merge 结果中 LoRA 验证没有残留 adapter 参几 ------------
merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

# CPU 清理占用空间的模型（下一次 baseline 加载很重）
del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

### 8.4 准备对照 baseline

把公开权重整理为与追加训练模型相同的输入 schema 和 processor。


In [ ]:
# ==============================================================
# 把比较 baseline 以与追加训练模型相同的 schema 重新保存
# ==============================================================
# 不直接评估公开初始权重，而是、与合并模型相同
# processor 配置・入输出 features 对齐为「对照模型」与并保存直。
# 这样评估时的预处理/和后处理完全部部一致、让比较公平。

# -済模型的 config 作为基础（入输出 schema 保持相同）
baseline_config = PreTrainedConfig.from_pretrained(
    MERGED_MODEL_DIR
)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 公开的初始 weight 读取同时、config 只将其与合并侧对齐
    baseline_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=baseline_config,
        strict=False,
    )

shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config 固定为单体模型运行格式（按与合并模型相同的方式处理）
baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

# processor stats 从合并侧复制（使两者完全部部一致）
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(
            source_path,
            BASELINE_MODEL_DIR / source_path.name,
        )

del baseline_policy
gc.collect()
torch.cuda.empty_cache()

print("Baseline ready.")

### 8.5 用 LeRobot subprocess 评估追加训练前后

用相同的 10 个任务和相同 seed 评估两个模型。每个模型 10 次 rollout，合计 20 次。


In [ ]:
# ==============================================================
# 追加训练前后的 2 模型以相同条条评估
# ==============================================================
# lerobot-eval  subprocess ， 2 次并、Section 5 ，仕込
# "EVAL_PROGRESS ..." 解析行并实时显示进度。
import json
import re
from collections import deque

# LIBERO 内部 camera 名称称称 → SmolVLA 映射到其预期名称称称
EVAL_CAMERA_MAPPING = {
    "agentview_image": "front",           # 俯视相机
    "robot0_eye_in_hand_image": "wrist",  # 手部相机
}


# lerobot-eval CLI 组装参几。-/合并模型共用的配置。
def build_eval_command(
    policy_path: Path,
    output_dir: Path,
) -> list[str]:
    return [
        "lerobot-eval",
        f"--policy.path={policy_path}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        "--env.task=libero_spatial",
        (
            "--env.task_ids="
            + json.dumps(
                EVAL_TASK_IDS,
                separators=(",", ":"),
            )
        ),
        (
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING,
                separators=(",", ":"),
            )
        ),
        # 观测图像 256x256（SmolVLA 匹配输入规格）
        "--env.observation_height=256",
        "--env.observation_width=256",
        # relative action 空间（LIBERO-plus 默认）
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",  # 顺序执行（VRAM 节省）
        "--eval.batch_size=1",
        (
            "--eval.n_episodes="
            f"{EVAL_EPISODES_PER_TASK}"
        ),
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={output_dir}",
    ]


# 1 运行两个模型的评估并 eval_info.json  dict 返次。
def run_evaluation(
    policy_path: Path,
    output_dir: Path,
    label: str,
) -> dict:
    shutil.rmtree(
        output_dir,
        ignore_errors=True,
    )

    # 传给评估子进程的环境变量
    eval_env = os.environ.copy()
    eval_env["MUJOCO_GL"] = "egl"
    # LIBERO-plus fork 和已打补丁的 LeRobot src 两者都放入优先路径
    eval_env["PYTHONPATH"] = (
        str(LIBERO_PLUS_DIR)
        + os.pathsep
        + str(LEROBOT_SRC)
        + os.pathsep
        + eval_env.get("PYTHONPATH", "")
    )
    eval_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_HUB_VERBOSITY"] = "error"
    eval_env["TQDM_DISABLE"] = "1"
    eval_env["PYTHONWARNINGS"] = "ignore"
    eval_env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        build_eval_command(policy_path, output_dir),
        cwd=LEROBOT_DIR,
        env=eval_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    # 失败转储用的最近日志
    recent_lines: deque[str] = deque(maxlen=120)

    # Section 5 ，仕込 "EVAL_PROGRESS task=x/y episode=a/b" 提取行的正则表达式
    progress_pattern = re.compile(
        r"^EVAL_PROGRESS "
        r"task=(\d+)/(\d+) "
        r"episode=(\d+)/(\d+)$"
    )

    # tqdm 输出全部部部 rollout 的进度
    from tqdm.auto import tqdm
    _total_rollouts = (
        len(EVAL_TASK_IDS) * EVAL_EPISODES_PER_TASK
    )
    _pbar_eval = tqdm(total=_total_rollouts, desc=f"eval[{label}]", unit="ep")

    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.replace("\r", "").strip()

        if not line:
            continue

        recent_lines.append(line)
        match = progress_pattern.match(line)

        if match:
            ti, tt, ei, et = match.groups()
            _pbar_eval.update(1)
            _pbar_eval.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")

    _pbar_eval.close()
    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "\n".join(recent_lines)
        )

    # 评估结果 JSON 读取并返次
    result_path = (
        output_dir
        / "eval_info.json"
    )

    if not result_path.is_file():
        raise FileNotFoundError(
            result_path
        )

    return json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


# 追加训练前的（processor 对齐）评估 baseline
BASE_EVAL_INFO = run_evaluation(
    BASELINE_MODEL_DIR,
    BASE_EVAL_DIR,
    "Base model",
)

# LoRA 合并模型用同样的 seed・相同 tasks ，评估
FINETUNED_EVAL_INFO = run_evaluation(
    MERGED_MODEL_DIR,
    FINETUNED_EVAL_DIR,
    "Spatial LoRA",
)

print("Evaluation complete.")

### 8.6 比较 LeRobot 版成功率

`Δ (pp)` 表示追加训练后的成功率减去追加训练前的成功率，单位是百分点。


In [ ]:
# ==============================================================
# 按任务的成功率和 Δ (pp) 汇总到表格
# ==============================================================
import pandas as pd
from IPython.display import display


# eval_info.json 的 per_task 从 task_id → 成功率(%) 创建。
def per_task_success(
    eval_info: dict,
) -> dict[int, float]:
    result: dict[int, float] = {}

    for task_info in eval_info["per_task"]:
        task_id = int(task_info["task_id"])
        successes = task_info["metrics"]["successes"]
        # bool 列表 → True 比示示示示例 %
        result[task_id] = (
            100.0
            * sum(bool(value) for value in successes)
            / len(successes)
        )

    return result


base_per_task = per_task_success(BASE_EVAL_INFO)
finetuned_per_task = per_task_success(
    FINETUNED_EVAL_INFO
)

# --- 每个 task 构建行 -----------------------------------------
rows = []

for task_id in EVAL_TASK_IDS:
    base_score = base_per_task[task_id]
    finetuned_score = finetuned_per_task[task_id]

    rows.append(
        {
            "Task ID": task_id,
            "Task": SPATIAL_TASK_NAMES[task_id],
            "Base (%)": base_score,
            "Spatial LoRA (%)": finetuned_score,
            # Δ (pp) 是「追加训练后 − 追加训练前」的百分点差
            "Δ (pp)": finetuned_score - base_score,
        }
    )

# --- Overall 行（LIBERO-Spatial 整体 pc_success） ------------
base_overall = float(
    BASE_EVAL_INFO["overall"]["pc_success"]
)
finetuned_overall = float(
    FINETUNED_EVAL_INFO["overall"]["pc_success"]
)

rows.append(
    {
        "Task ID": "Overall",
        "Task": "LIBERO-Spatial",
        "Base (%)": base_overall,
        "Spatial LoRA (%)": finetuned_overall,
        "Δ (pp)": finetuned_overall - base_overall,
    }
)

# DataFrame 化并 CSV 同时降维到 notebook 中显示
comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
)

display(comparison_df.round(1))

print(
    f"Overall: {base_overall:.1f}% → "
    f"{finetuned_overall:.1f}% "
    f"({finetuned_overall - base_overall:+.1f} pp)"
)

### 8.7 将合并模型压缩为 zip


In [ ]:
# ==============================================================
# 合并模型 zip 打包以便取出
# ==============================================================
# Colab ，是 files.download 通过浏览器将 zip 下载.
# notebook 结束时 /content/ 会消失，因此、务必复制到外部存储.
from zipfile import ZIP_STORED, ZipFile
from google.colab import files

# 如果已有 zip 则删除（为重复执行）
if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

# ZIP_STORED（不压缩）之所以选择，是因为 safetensors 已经压缩，、
# 重新次压缩也不会变小，。IO 只会增加时间，所以无需压缩。
with ZipFile(
    MERGED_ZIP_PATH,
    mode="w",
    compression=ZIP_STORED,
    allowZip64=True,  # safetensors  4GB 超过该大小时的保险
) as archive:
    for file_path in sorted(
        MERGED_MODEL_DIR.rglob("*")
    ):
        if file_path.is_file():
            # zip 内部 MERGED_MODEL_DIR 的 basename  root 
            archive.write(
                file_path,
                arcname=(
                    Path(MERGED_MODEL_DIR.name)
                    / file_path.relative_to(
                        MERGED_MODEL_DIR
                    )
                ),
            )

print(f"Saved: {MERGED_ZIP_PATH}")
files.download(str(MERGED_ZIP_PATH))
files.download(str(COMPARISON_CSV_PATH))

### 8.8 LeRobot 版 rollout 视频（可选）


In [ ]:
# ==============================================================
# 视频 cell: -済模型的 rollout  mp4 查看
# ==============================================================
# 手順:
#   1. lerobot_eval.py 录制几量 LEROBOT_MAX_RECORDED 通过环境变量
#      以便控制 idempotent 打补丁 (Section 5 扩展补丁)
#   2. 1 任务 × N 个 rollout  recording=true ，执行
#   3. ， mp4  notebook 嵌入显示
from IPython.display import Video, display

VIDEO_TASK_ID = 0          # LIBERO-Spatial 的 task 0 录制 (其他 task 若 0〜9)
VIDEO_N_EPISODES = 2       # 录制几量。mp4  notebook 会嵌入 notebook，请勿设置过大
VIDEO_OUT_DIR = WORKDIR / "eval" / "video"

# --- 1. eval_script  env-var 应用兼容补丁 (idempotent) --------
# Section 5 加入的 "max_episodes_rendered = 0" (固定) 
# 环境变量 LEROBOT_MAX_RECORDED 替换为可通过环境变量切换的形式。
eval_script_path = (
    LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
)
_src = eval_script_path.read_text(encoding="utf-8")

# eval_script 是 module top ， os  import 尚未执行，因此
# __import__("os") 使用 self-contained 写成官方文档的
_flexible_patch = (
    'max_episodes_rendered = '
    'int(__import__("os").environ.get("LEROBOT_MAX_RECORDED", "0"))'
)
_broken_patch = (
    'max_episodes_rendered = '
    'int(os.environ.get("LEROBOT_MAX_RECORDED", "0"))'
)
if _flexible_patch not in _src:
    if _broken_patch in _src:
        # 过去如果有 broken 版个个个则修正
        _src = _src.replace(_broken_patch, _flexible_patch, 1)
    else:
        _old_patch = "max_episodes_rendered = 0"
        if _old_patch not in _src:
            raise RuntimeError(
                "eval_script 找不到补丁位置。"
                "Section 5 请先执行。"
            )
        _src = _src.replace(_old_patch, _flexible_patch, 1)
    eval_script_path.write_text(_src, encoding="utf-8")

# --- 2. 带录制的 eval 执行 ------------------------------------
shutil.rmtree(VIDEO_OUT_DIR, ignore_errors=True)

video_env = os.environ.copy()
video_env["MUJOCO_GL"] = "egl"
video_env["PYTHONPATH"] = (
    str(LIBERO_PLUS_DIR)
    + os.pathsep
    + str(LEROBOT_SRC)
    + os.pathsep
    + video_env.get("PYTHONPATH", "")
)
video_env["LEROBOT_MAX_RECORDED"] = str(VIDEO_N_EPISODES)
video_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
video_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
video_env["HF_HUB_VERBOSITY"] = "error"
video_env["TQDM_DISABLE"] = "1"
video_env["PYTHONWARNINGS"] = "ignore"
video_env["PYTHONUNBUFFERED"] = "1"

video_command = [
    "lerobot-eval",
    f"--policy.path={MERGED_MODEL_DIR}",
    "--policy.device=cuda",
    "--policy.use_amp=false",
    "--env.type=libero",
    "--env.is_libero_plus=true",
    "--env.task=libero_spatial",
    "--env.task_ids="
    + json.dumps([VIDEO_TASK_ID], separators=(",", ":")),
    "--env.camera_name_mapping="
    + json.dumps(EVAL_CAMERA_MAPPING, separators=(",", ":")),
    "--env.observation_height=256",
    "--env.observation_width=256",
    "--env.control_mode=relative",
    "--env.max_parallel_tasks=1",
    "--eval.batch_size=1",
    f"--eval.n_episodes={VIDEO_N_EPISODES}",
    "--eval.use_async_envs=false",
    # recording=false + LEROBOT_MAX_RECORDED>0 的组合会、
    # eval_script  videos_dir 到 mp4 进入输出路径 (recording=true 是
    # HF dataset 用于完整记录格式， mp4 与其互斥).
    "--eval.recording=false",
    f"--seed={EVAL_SEED}",
    f"--output_dir={VIDEO_OUT_DIR}",
]

print(
    f"Recording {VIDEO_N_EPISODES} rollout(s) on task {VIDEO_TASK_ID}...",
    flush=True,
)
_video_process = subprocess.Popen(
    video_command,
    cwd=LEROBOT_DIR,
    env=video_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert _video_process.stdout is not None
_video_recent: deque[str] = deque(maxlen=120)
_prog_pat = re.compile(
    r"^EVAL_PROGRESS "
    r"task=(\d+)/(\d+) "
    r"episode=(\d+)/(\d+)$"
)
for _raw in _video_process.stdout:
    _line = _raw.replace("\r", "").strip()
    if not _line:
        continue
    _video_recent.append(_line)
    _m = _prog_pat.match(_line)
    if _m:
        _, _, ei, et = _m.groups()
        print(f"  episode {ei}/{et}", flush=True)
_rc = _video_process.wait()
if _rc != 0:
    print("\n".join(_video_recent))
    raise RuntimeError(f"video eval failed: {_rc}")

# --- 3. ， mp4  notebook 嵌入 ------------------------
mp4_files = sorted(VIDEO_OUT_DIR.rglob("*.mp4"))
if not mp4_files:
    raise RuntimeError(
        f"mp4  {VIDEO_OUT_DIR} 中不存在。"
    )

print(f"\nFound {len(mp4_files)} mp4 file(s):")
for _mp4 in mp4_files:
    print(f"  {_mp4.relative_to(VIDEO_OUT_DIR)}")
    # embed=True 设为后 base64 嵌入为 notebook 可以单独播放
    # (但文条会变大)。如果只想引用 embed=False 到。
    display(Video(str(_mp4), embed=True, html_attributes="controls loop"))

## 9. 进阶评估：LIBERO-plus 4 suite × 3 task × 1 episode

本 section 比 Section 8 的 Spatial 小评估覆盖更广：对 Spatial、Object、Goal 和 Long-horizon 四个 suite，各选 3 个任务、每个 1 个 episode，对**追加训练后的模型和对照 baseline**各执行 12 次 rollout。

### 输入格式契约

本 cell 假定 `MERGED_MODEL_DIR` 和 `BASELINE_MODEL_DIR` 都是由 `SmolVLAPolicy.save_pretrained` 生成的可加载目录。不要手动重命名配置键、processor 文件或模型权重；评估会同时读取两套目录，并使用同一组环境和随机种子。

### 选择逻辑与解读

每个 suite 选择 3 个任务，优先覆盖训练域、物体泛化、目标语言泛化和长时序泛化。每个模型共 12 个 episode，所有结果会汇总为 suite 与总体成功率。

这只是轻量广域 smoke test，不是完整 benchmark。重型 CUDA、网络下载、Drive 和第三方资源未在本轮全部验证时，不应把结果表述为端到端复现实验。


In [ ]:
# ==============================================================
# 9.0 Preflight check: section 9 是否满足要求的输入格式
# ==============================================================
# section 8 自己尝试改进后的结果、模型目录格式是否
# 损坏，先行确认. 若此处失败， section 8 重新检查.

_REQUIRED_FILES = [
    "model.safetensors",
    "config.json",
    "policy_preprocessor.json",
    "policy_postprocessor.json",
]
# .safetensors 的 stats 名称称称可变，因此 glob 验证
_REQUIRED_GLOBS = [
    "policy_preprocessor*.safetensors",
    "policy_postprocessor*.safetensors",
]

def _check_policy_dir(policy_dir: Path, label: str) -> None:
    print(f"\n[{label}] {policy_dir}")
    if not policy_dir.is_dir():
        raise FileNotFoundError(
            f"{label}: 目录不存在。"
            "section 8 是否执行了对应 cell?"
        )
    missing = []
    for name in _REQUIRED_FILES:
        p = policy_dir / name
        ok = p.is_file()
        print(f"  {'✅' if ok else '❌'}  {name}"
              + (f"  ({p.stat().st_size / 1e6:.1f} MB)" if ok else ""))
        if not ok:
            missing.append(name)
    for pattern in _REQUIRED_GLOBS:
        matches = list(policy_dir.glob(pattern))
        ok = len(matches) > 0
        print(f"  {'✅' if ok else '❌'}  {pattern}  ({len(matches)} match)")
        if not ok:
            missing.append(pattern)
    if missing:
        raise FileNotFoundError(
            f"{label} 缺少需要文条: {missing}. "
            "section 8.3 (merge) / 8.4 (baseline) 请重新执行。"
        )


_check_policy_dir(BASELINE_MODEL_DIR, "BASELINE_MODEL_DIR (对照)")
_check_policy_dir(MERGED_MODEL_DIR, "MERGED_MODEL_DIR (改进版个个)")
print("\n✅ 契约 OK — section 9 可以执行进阶评估")

In [ ]:
# ==============================================================
# 进阶评估: 4 suite × 3 task × 1 ep = 12 rollout / 模型
# ==============================================================
# 每个任务选择 suite 的 10 从任务中等间隔选择 3 个任务、
# 「相同 task 重复」比「不同的 task 广泛运行」优先多样性的策略.
# 1 每个模型 12 rollout, 2 模型 (base + finetuned) 合计 24 rollout.
#
# 内分:
#   libero_spatial : 3 tasks (训练域)
#   libero_object  : 3 tasks (物体泛化)
#   libero_goal    : 3 tasks (目标词汇泛化)
#   libero_10      : 3 tasks (long-horizon 泛化)

# 每个 suite 的 10 tasks (id 0-9) 从等间隔到 3 个样个 (start / middle / end)
_ADV_TASK_IDS = [0, 4, 8]

ADVANCED_SUITES = {
    "libero_spatial": _ADV_TASK_IDS,
    "libero_object":  _ADV_TASK_IDS,
    "libero_goal":    _ADV_TASK_IDS,
    "libero_10":      _ADV_TASK_IDS,
}
ADVANCED_EPISODES_PER_TASK = 1
ADVANCED_OUT_DIR = WORKDIR / "eval" / "advanced"


def run_advanced_eval(policy_path, out_root, label):
    """1 对模型全部部部 suite 运行 suite→eval_info 的 dict 返次。"""
    shutil.rmtree(out_root, ignore_errors=True)
    suite_results = {}
    for suite_name, task_ids in ADVANCED_SUITES.items():
        sub_dir = out_root / suite_name

        adv_env = os.environ.copy()
        adv_env["MUJOCO_GL"] = "egl"
        adv_env["PYTHONPATH"] = (
            str(LIBERO_PLUS_DIR)
            + os.pathsep
            + str(LEROBOT_SRC)
            + os.pathsep
            + adv_env.get("PYTHONPATH", "")
        )
        adv_env["LEROBOT_MAX_RECORDED"] = "0"
        adv_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        adv_env["TQDM_DISABLE"] = "1"
        adv_env["PYTHONWARNINGS"] = "ignore"
        adv_env["PYTHONUNBUFFERED"] = "1"

        adv_command = [
            "lerobot-eval",
            f"--policy.path={policy_path}",
            "--policy.device=cuda",
            "--policy.use_amp=false",
            "--env.type=libero",
            "--env.is_libero_plus=true",
            f"--env.task={suite_name}",
            "--env.task_ids="
            + json.dumps(task_ids, separators=(",", ":")),
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING, separators=(",", ":")
            ),
            "--env.observation_height=256",
            "--env.observation_width=256",
            "--env.control_mode=relative",
            "--env.max_parallel_tasks=1",
            "--eval.batch_size=1",
            f"--eval.n_episodes={ADVANCED_EPISODES_PER_TASK}",
            "--eval.use_async_envs=false",
            "--eval.recording=false",
            f"--seed={EVAL_SEED}",
            f"--output_dir={sub_dir}",
        ]
        print(
            f"\n[{label}] {suite_name} "
            f"({len(task_ids)} tasks × "
            f"{ADVANCED_EPISODES_PER_TASK} ep)",
            flush=True,
        )
        proc = subprocess.Popen(
            adv_command,
            cwd=LEROBOT_DIR,
            env=adv_env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert proc.stdout is not None
        adv_recent: deque[str] = deque(maxlen=120)
        _prog = re.compile(
            r"^EVAL_PROGRESS "
            r"task=(\d+)/(\d+) "
            r"episode=(\d+)/(\d+)$"
        )

        # tqdm 在这里输出该 suite 的 rollout 输出进度
        from tqdm.auto import tqdm
        _total_this_suite = len(task_ids) * ADVANCED_EPISODES_PER_TASK
        _pbar_adv = tqdm(
            total=_total_this_suite,
            desc=f"{label} {suite_name}",
            unit="ep",
        )

        for raw in proc.stdout:
            line = raw.replace("\r", "").strip()
            if not line:
                continue
            adv_recent.append(line)
            m = _prog.match(line)
            if m:
                ti, tt, ei, et = m.groups()
                _pbar_adv.update(1)
                _pbar_adv.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")

        _pbar_adv.close()
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError("\n".join(adv_recent))
        info = json.loads(
            (sub_dir / "eval_info.json").read_text(
                encoding="utf-8"
            )
        )
        suite_results[suite_name] = info
    return suite_results


print("=== Advanced eval: baseline ===")
adv_base = run_advanced_eval(
    BASELINE_MODEL_DIR,
    ADVANCED_OUT_DIR / "baseline",
    "Base",
)

print("\n=== Advanced eval: fine-tuned ===")
adv_ft = run_advanced_eval(
    MERGED_MODEL_DIR,
    ADVANCED_OUT_DIR / "finetuned",
    "LoRA",
)


# --- 全部部 rollout 总体成功率 ---------------------------------
def overall_across_suites(results):
    total_success = 0.0
    total_rollouts = 0
    for info in results.values():
        for task in info["per_task"]:
            successes = task["metrics"]["successes"]
            total_success += sum(bool(v) for v in successes)
            total_rollouts += len(successes)
    return (
        100.0 * total_success / total_rollouts
        if total_rollouts
        else 0.0
    )


# --- suite 按 suite 的汇总表 ----------------------------------
adv_rows = []
for suite_name in ADVANCED_SUITES:
    b = adv_base[suite_name]["overall"]["pc_success"]
    f = adv_ft[suite_name]["overall"]["pc_success"]
    adv_rows.append({
        "Suite": suite_name,
        "Tasks": len(ADVANCED_SUITES[suite_name]),
        "Base (%)": b,
        "Spatial LoRA (%)": f,
        "Δ (pp)": f - b,
    })

adv_base_overall = overall_across_suites(adv_base)
adv_ft_overall = overall_across_suites(adv_ft)
adv_rows.append({
    "Suite": "Overall (12 rollouts)",
    "Tasks": sum(len(v) for v in ADVANCED_SUITES.values()),
    "Base (%)": adv_base_overall,
    "Spatial LoRA (%)": adv_ft_overall,
    "Δ (pp)": adv_ft_overall - adv_base_overall,
})

adv_df = pd.DataFrame(adv_rows)
adv_csv = ADVANCED_OUT_DIR / "advanced_comparison.csv"
adv_df.to_csv(adv_csv, index=False)
display(adv_df.round(1))
print(f"\nSaved: {adv_csv}")
print(
    f"Overall (12 rollouts): {adv_base_overall:.1f}% -> "
    f"{adv_ft_overall:.1f}% "
    f"({adv_ft_overall - adv_base_overall:+.1f} pp)"
)